# Addestramento del filtro 4B dell'Assistente PM (Projexa)

Questo notebook addestra **pm-projexa-filtro** (Qwen3 4B Instruct, qwen3:4b-instruct): il modello piccolo e veloce che, durante la call con la voce **«Projexa 4B + 8B»**, legge ogni pezzo di riunione e decide se passarlo a Pm projexa (8B) per scrivere il suggerimento. Risponde solo **«SI n»** (c'è il segnale n) o **«NO»**.

Impara dalle stesse valutazioni della pagina **Analisi PM**: pezzo con domanda Corretta = «SI n», pezzo con domande solo Errate o senza domanda = «NO».
Il risultato è un piccolo file **adattatore** (`pm-projexa-filtro-adapter.gguf`, 50-150 MB) da caricare in Analisi PM › **Modello sulla VM** scegliendo **pm-projexa-filtro**.

**Prima di iniziare**
1. Menu **Runtime → Cambia tipo di runtime → GPU T4** (gratuita) → Salva.
2. Tieni a portata il file `pm_projexa_filtro_addestramento_AAAA-MM-GG.jsonl` scaricato da Analisi PM (pulsante del **filtro**, non quello di Pm projexa).

**Poi** menu **Runtime → Esegui tutto**. Alla cella 2 si apre il pulsante per caricare il file .jsonl.
Tempo indicativo: 15-40 minuti. Alla fine il file .gguf si scarica da solo.

Se una cella dà errore, copia il messaggio e mandalo a chi ti segue lo sviluppo: le librerie cambiano spesso versione.


## 1. Installazione delle librerie (5-10 minuti)


In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), 'Nessuna GPU: Runtime > Cambia tipo di runtime > GPU T4'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Carica il file degli esempi del filtro (.jsonl scaricato da Projexa)
Una parte degli esempi (10%, al massimo 40) resta da parte per la prova finale: il modello non la vede durante l'addestramento.


In [ ]:
from google.colab import files
import json, random
caricati = files.upload()
nome = next(n for n in caricati if n.endswith('.jsonl'))
esempi = [json.loads(r) for r in open(nome, encoding='utf-8') if r.strip()]
risposta = lambda e: e['messages'][-1]['content'].strip().upper()
assert all(risposta(e).startswith(('SI', 'NO')) for e in esempi), 'Questo non è il file del filtro (le risposte devono essere SI n / NO): scarica quello del filtro da Analisi PM'
si = sum(1 for e in esempi if risposta(e).startswith('SI'))
print(f'{len(esempi)} esempi: {si} SI, {len(esempi) - si} NO')
if si < 50:
    print('ATTENZIONE: meno di 50 esempi di SI, il filtro imparerà poco. Meglio aspettare di avere più domande corrette.')
random.Random(42).shuffle(esempi)
n_prova = min(40, len(esempi) // 10) if len(esempi) >= 40 else 0
prova, addestra = esempi[:n_prova], esempi[n_prova:]
print(f'{len(addestra)} esempi per addestrare, {len(prova)} per la prova finale')


## 3. Modello base (Qwen3 4B Instruct, lo stesso del filtro sulla VM)


In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3-4B-Instruct-2507', max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)


## 4. Addestramento
Il modello impara solo la risposta (SI n / NO), non a riscrivere la trascrizione.


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
testi = [tokenizer.apply_chat_template(e['messages'], tokenize=False) for e in addestra]
dati = Dataset.from_dict({'text': testi}).shuffle(seed=42)
epoche = 3 if len(addestra) < 400 else 2
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dati,
    args=SFTConfig(dataset_text_field='text', max_seq_length=2048, per_device_train_batch_size=2,
                   gradient_accumulation_steps=4, num_train_epochs=epoche, learning_rate=2e-4,
                   warmup_steps=5, logging_steps=5, optim='adamw_8bit', fp16=True, bf16=False,
                   output_dir='uscita', report_to='none', seed=42))
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
except Exception as e:
    print('Addestramento su tutto il testo (train_on_responses_only non disponibile):', e)
trainer.train()


## 5. Prova sugli esempi tenuti da parte
**SI riconosciuti** è il numero più importante: un SI perso è un suggerimento che Pm projexa non scriverà mai. Un NO sbagliato costa solo qualche secondo di lavoro in più all'8B.


In [ ]:
FastLanguageModel.for_inference(model)
def chiedi(messaggi):
    ids = tokenizer.apply_chat_template(messaggi, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=6, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
si_tot = si_presi = no_tot = no_giusti = 0
for e in prova:
    r = chiedi(e['messages'][:-1])
    atteso_si = risposta(e).startswith('SI')
    detto_si = not r.upper().startswith('NO')  # come sulla VM: risposta non chiara = passa all'8B
    if atteso_si:
        si_tot += 1; si_presi += detto_si
    else:
        no_tot += 1; no_giusti += not detto_si
if prova:
    print(f'SI riconosciuti: {si_presi}/{si_tot} · NO riconosciuti: {no_giusti}/{no_tot}')
else:
    print('Pochi esempi: prova finale saltata.')


## 6. Adattatore in formato Ollama (.gguf) e download


In [ ]:
model.save_pretrained('adattatore')
tokenizer.save_pretrained('adattatore')
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py adattatore --base-model-id Qwen/Qwen3-4B-Instruct-2507 --outtype f16 --outfile pm-projexa-filtro-adapter.gguf
import os
print('Dimensione:', round(os.path.getsize('pm-projexa-filtro-adapter.gguf') / 1e6), 'MB')
files.download('pm-projexa-filtro-adapter.gguf')


## 7. Fatto

Carica `pm-projexa-filtro-adapter.gguf` in Projexa: **Analisi PM → Modello sulla VM**, scegli **pm-projexa-filtro**, poi **Carica e installa**.
Projexa fa il backup del filtro attuale (se c'è), installa il nuovo sopra la base qwen3:4b-instruct e lo prova; se qualcosa non va torna da solo al precedente.
Dalla call successiva la voce **«Projexa 4B + 8B»** usa il filtro addestrato.
